# Canadian Pipeline Dataset — Cleaning and Standardization

This notebook reads pipeline datasets from different source formats, maps their fields into a common schema, normalizes geometry, combines records, removes identified duplicates, and exports a standardized CSV.

## Workflow
1. Install and import the required libraries.
2. Configure the project and output folders.
3. Define the standardized schema and source-field aliases.
4. Define reusable file-reading, field-normalization, and geometry helpers.
5. Clean each source dataset and combine the results.
6. Load source files and run the cleaning pipeline.
7. Inspect and download the cleaned output.

**Supported inputs:** CSV, TXT, Excel (`.xlsx`, `.xls`), GeoJSON, and JSON.

**Colab note:** this notebook uses `/content` as its working directory by default. Upload files or mount Google Drive before running the cleaning step.


## 1. Install dependencies

GeoPandas and Shapely are used to read spatial files and parse geometry.

In [ ]:
%pip -q install geopandas openpyxl shapely


## 2. Imports and project folders

In a notebook, `__file__` is not defined. This version uses the current working directory as the project folder, which works in Google Colab.

In [ ]:
import json
import re
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely import wkt


# --------------------------------------------------
# PROJECT DIRECTORIES
# --------------------------------------------------

# Use the current notebook working directory (e.g., /content in Colab).
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project directory: {BASE_DIR}")
print(f"Data directory: {DATA_DIR}")


## 3. Define the standardized dataset schema

Every source dataset is mapped into these columns. Missing source attributes are represented as null values.

In [ ]:
STANDARD_COLUMNS = [
    "pipeline_name",
    "company",
    "parent_company",
    "province",
    "location_description",
    "from_location",
    "to_location",
    "commodity",
    "pipeline_length_km",
    "capacity",
    "status",
    "start_date",
    "end_date",
    "segment_number",
    "licence_number",
    "project_number",
    "authority_type",
    "data_source",
    "latitude",
    "longitude",
    "geometry",
    "source_dataset",
    "source_record_id",
    "source_fields",
]


## 4. Map source column names to standard fields

Aliases account for naming differences across source datasets. Column names are normalized before matching.

In [ ]:
ALIASES = {
    "pipeline_name": [
        "pipeline name", "name", "name_en", "name_fr"
    ],
    "company": [
        "cer-regulated company", "company_name", "proponent", "company"
    ],
    "parent_company": [
        "parent company"
    ],
    "province": [
        "province", "province_name"
    ],
    "location_description": [
        "location (description)", "location", "location_description"
    ],
    "from_location": [
        "from_location", "from location"
    ],
    "to_location": [
        "to_location", "to location"
    ],
    "commodity": [
        "transported commodity", "commodity", "substance 1", "line_type_desc"
    ],
    "pipeline_length_km": [
        "cer-regulated pipeline length (km)", "pipeline_length_km"
    ],
    "capacity": [
        "pipeline capacity", "capacity"
    ],
    "status": [
        "status", "segment_status"
    ],
    "start_date": [
        "commenced operations", "start_date", "approval_date", "approval date"
    ],
    "end_date": [
        "end_date", "retirement_date"
    ],
    "segment_number": [
        "segment_number"
    ],
    "licence_number": [
        "licence_number", "license_number"
    ],
    "project_number": [
        "project_number"
    ],
    "authority_type": [
        "authority_type"
    ],
    "data_source": [
        "data_source"
    ],
}


## 5. File-reading and normalization helpers

These functions normalize column names and read supported source formats. CSV reading tries common encodings to handle files that are not UTF-8.

In [ ]:
def normalize_column(name):
    """Convert a column name to lowercase snake_case."""
    name = str(name).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")


def read_csv_safely(path):
    """Read a CSV file using common encodings and pandas' C parser."""
    last_error = None

    for encoding in ("utf-8", "cp1252", "latin1"):
        try:
            return pd.read_csv(
                path,
                sep=",",
                encoding=encoding,
                engine="c",
                low_memory=False,
                on_bad_lines="skip",
            )
        except UnicodeDecodeError as error:
            last_error = error

    raise ValueError(
        f"Could not read {path}. Last encoding error: {last_error}"
    )


def read_file(path):
    """Read CSV, TXT, Excel, GeoJSON, or JSON input files."""
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix in (".geojson", ".json"):
        return gpd.read_file(path)

    if suffix in (".xlsx", ".xls"):
        return pd.read_excel(path)

    if suffix in (".csv", ".txt"):
        return read_csv_safely(path)

    raise ValueError(f"Unsupported file format: {path.suffix}")


## 6. Geometry parsing and validation

Geometry values may arrive as WKT strings, Shapely objects, or missing values. The helper functions normalize these inputs to valid, non-empty Shapely geometries or `None`.

In [ ]:
def parse_wkt_geometry(value):
    """Parse a WKT string; return None if it is missing or invalid."""
    if not isinstance(value, str) or not value.strip():
        return None

    try:
        return wkt.loads(value)
    except Exception:
        return None


def is_valid_geometry(geom):
    """Check that a value is a non-empty Shapely geometry."""
    return (
        geom is not None
        and hasattr(geom, "geom_type")
        and hasattr(geom, "is_empty")
        and not geom.is_empty
    )


def normalize_geometry_value(value):
    """Return a valid Shapely geometry or None for a cell value."""
    if is_valid_geometry(value):
        return value

    if isinstance(value, str):
        parsed = parse_wkt_geometry(value)
        return parsed if is_valid_geometry(parsed) else None

    return None


def get_geometries(df):
    """Find and normalize the first recognized geometry column."""
    geometry_values = None

    if isinstance(df, gpd.GeoDataFrame):
        geometry_values = df.geometry
    else:
        for column in ("the_geom", "geom", "wkt", "geometry"):
            if column in df.columns:
                geometry_values = df[column]
                break

    if geometry_values is None:
        return pd.Series(None, index=df.index, dtype=object)

    return geometry_values.apply(normalize_geometry_value)


## 7. Source identifiers and original-field preservation

A source record ID is selected from a prioritized list of common identifier fields. Original non-geometry attributes are serialized to JSON so source information remains available after standardization.

In [ ]:
def get_source_record_id(df):
    """Find the best available source record identifier."""
    id_fields = [
        "objectid",
        "pinfo_id",
        "licence_number",
        "project_number",
        "segment_number",
    ]

    result = pd.Series(None, index=df.index, dtype=object)

    for column in id_fields:
        if column in df.columns:
            result = result.fillna(df[column])

    return result


def serialize_source_fields(df):
    """Serialize original non-geometry fields as JSON strings."""
    geometry_columns = {"geometry", "the_geom", "geom", "wkt"}
    source_columns = [
        column for column in df.columns
        if column not in geometry_columns
    ]

    records = df[source_columns].to_dict(orient="records")

    return [
        json.dumps(record, ensure_ascii=False, default=str)
        for record in records
    ]


## 8. Main dataset-cleaning function

The function below processes each source dataset independently, maps source attributes to the standard schema, normalizes geometries, extracts representative-point coordinates, and combines the cleaned results.

It then:
- Converts length and coordinate columns to numeric values.
- Removes duplicate records when a source identifier is available.
- Trims leading and trailing whitespace in object columns.
- Exports the combined dataset as UTF-8-with-BOM CSV.


In [ ]:
def clean_pipeline_dataset(
    file_paths,
    output_csv="Canadian_Pipelines_Clean.csv",
):
    """
    Combine pipeline datasets into one standardized CSV.

    Parameters
    ----------
    file_paths : dict
        Dictionary mapping dataset names to input file paths.
    output_csv : str or pathlib.Path, default "Canadian_Pipelines_Clean.csv"
        Output filename. Relative paths are saved inside DATA_DIR.

    Returns
    -------
    pandas.DataFrame
        Combined and cleaned pipeline dataset.
    """
    output_path = Path(output_csv)

    # Place relative output paths inside the configured data directory.
    if not output_path.is_absolute():
        output_path = DATA_DIR / output_path

    output_path.parent.mkdir(parents=True, exist_ok=True)

    all_records = []

    # Process each source dataset separately.
    for dataset_name, file_path in file_paths.items():
        print(f"\nReading {dataset_name}...")

        df = read_file(file_path)

        # Standardize source column names before matching aliases.
        df.columns = [normalize_column(column) for column in df.columns]

        print(f"Loaded {len(df):,} records.")

        cleaned = pd.DataFrame(index=df.index)

        # Map available source fields to the standard dataset schema.
        for standard_field, aliases in ALIASES.items():
            matching_column = next(
                (
                    normalize_column(alias)
                    for alias in aliases
                    if normalize_column(alias) in df.columns
                ),
                None,
            )

            cleaned[standard_field] = (
                df[matching_column] if matching_column is not None else None
            )

        # Preserve source provenance and record identifiers.
        cleaned["source_dataset"] = dataset_name
        cleaned["source_record_id"] = get_source_record_id(df)

        # Parse geometries once, then reuse them for WKT and coordinates.
        geometries = get_geometries(df)

        # Store geometries as WKT strings for CSV compatibility.
        cleaned["geometry"] = geometries.apply(
            lambda geom: geom.wkt if is_valid_geometry(geom) else None
        )

        # Use each geometry's representative point for a stable coordinate pair.
        def representative_coordinate(geom, coordinate):
            if not is_valid_geometry(geom):
                return None

            point = geom.representative_point()
            return point.y if coordinate == "latitude" else point.x

        cleaned["latitude"] = geometries.apply(
            lambda geom: representative_coordinate(geom, "latitude")
        )
        cleaned["longitude"] = geometries.apply(
            lambda geom: representative_coordinate(geom, "longitude")
        )

        # Retain original non-geometry attributes for traceability.
        cleaned["source_fields"] = serialize_source_fields(df)

        # Ensure every expected output field exists and follows the same order.
        for column in STANDARD_COLUMNS:
            if column not in cleaned.columns:
                cleaned[column] = None

        cleaned = cleaned[STANDARD_COLUMNS]

        # Drop rows that contain no useful identifying information.
        useful_columns = [
            "pipeline_name",
            "company",
            "geometry",
            "location_description",
            "from_location",
            "licence_number",
            "project_number",
        ]

        cleaned = cleaned.dropna(how="all", subset=useful_columns)
        all_records.append(cleaned)

        print(f"Processed {len(cleaned):,} records from {dataset_name}.")

    if not all_records:
        raise ValueError("No datasets were supplied in file_paths.")

    # Combine the standardized records from all input datasets.
    print("\nCombining datasets...")
    combined = pd.concat(all_records, ignore_index=True)

    # Convert selected fields to numeric values; invalid entries become NaN.
    for column in ("pipeline_length_km", "latitude", "longitude"):
        combined[column] = pd.to_numeric(
            combined[column],
            errors="coerce",
        )

    # Deduplicate only records that have a source ID. Include dataset name
    # because identifiers may be reused across different source datasets.
    has_id = combined["source_record_id"].notna()

    identified = combined.loc[has_id].drop_duplicates(
        subset=["source_dataset", "source_record_id"],
        keep="first",
    )
    unidentified = combined.loc[~has_id]

    combined = pd.concat(
        [identified, unidentified],
        ignore_index=True,
    )

    # Trim whitespace from string values without changing non-string values.
    for column in combined.select_dtypes(include="object").columns:
        combined[column] = combined[column].apply(
            lambda value: value.strip() if isinstance(value, str) else value
        )

    # Export the final standardized dataset.
    combined.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )

    print("\n----------------------------------")
    print("PIPELINE DATA CLEANING COMPLETE")
    print("----------------------------------")
    print(f"Total records: {len(combined):,}")
    print(f"Output file: {output_path.resolve()}")
    print("----------------------------------")

    return combined


## 9. Specify the input datasets

Create a dictionary where each key is a descriptive dataset name and each value is the path to that dataset.

### Google Colab options
- Upload files using the Files panel, then reference their paths (often `/content/filename.csv`).
- Or mount Google Drive and use paths under `/content/drive/MyDrive/`.

The example entries below are placeholders. Replace them with your actual files before running the next cell.


In [ ]:
# Example: replace these entries with your actual input dataset paths.
file_paths = {
    # "CER_Pipelines": "/content/CER_Pipelines.csv",
    # "Provincial_Pipelines": "/content/Provincial_Pipelines.geojson",
    # "Other_Source": "/content/Other_Source.xlsx",
}

if not file_paths:
    print("Add at least one dataset path to `file_paths` before continuing.")
else:
    print(f"Datasets configured: {len(file_paths)}")
    for dataset_name, file_path in file_paths.items():
        print(f"  - {dataset_name}: {file_path}")


## 10. Run the cleaning workflow

This cell validates that input paths have been configured, then calls the main cleaning function.

In [ ]:
if not file_paths:
    raise ValueError(
        "No input datasets configured. Add dataset names and file paths "
        "in the previous cell, then rerun this cell."
    )

cleaned_pipelines_df = clean_pipeline_dataset(
    file_paths=file_paths,
    output_csv="Canadian_Pipelines_Clean.csv",
)

display(cleaned_pipelines_df.head())


## 11. Inspect the cleaned dataset

Review the output dimensions, available columns, missing geometry count, and source dataset distribution.

In [ ]:
print("Dataset shape:", cleaned_pipelines_df.shape)

print("\nMissing geometry values:")
print(cleaned_pipelines_df["geometry"].isna().sum())

print("\nRecords by source dataset:")
display(
    cleaned_pipelines_df["source_dataset"]
    .value_counts(dropna=False)
    .rename_axis("source_dataset")
    .to_frame("record_count")
)

print("\nColumn data types:")
display(cleaned_pipelines_df.dtypes.to_frame("dtype"))


## 12. Download the cleaned CSV in Google Colab

The CSV is saved in the notebook's `data/` folder. The following cell provides a Colab download prompt.

In [ ]:
output_csv_path = DATA_DIR / "Canadian_Pipelines_Clean.csv"

print(f"Saved file: {output_csv_path.resolve()}")

try:
    from google.colab import files
    files.download(str(output_csv_path))
except ImportError:
    print("Not running in Google Colab. Retrieve the file from the data folder.")


## Notes and assumptions

- **Coordinate reference system:** coordinates are extracted directly from geometry coordinates. If a source geometry uses a projected CRS rather than longitude/latitude, reproject it to EPSG:4326 before interpreting the extracted values as geographic coordinates.
- **Geometry validation:** this workflow rejects missing and empty geometries, but does not automatically repair topologically invalid geometries.
- **CSV parsing:** malformed CSV rows are skipped by `on_bad_lines="skip"`. Review source files if row-level completeness is important.
- **Deduplication:** records are deduplicated only when `source_record_id` exists, using the combination of source dataset and ID.
- **Source fields:** original non-geometry attributes are retained in the `source_fields` JSON column for traceability.
